# PlantDoc AI — Training MobileNetV2 di Google Colab

Klasifikasi 4 kondisi daun tomat: **Healthy, Early Blight, Late Blight, Leaf Mold**.

Alur: dataset PlantVillage → split 70/15/15 → transfer learning MobileNetV2 → evaluasi → ekspor model + peta label.

**Sebelum mulai:**
1. Buka notebook ini di [colab.research.google.com](https://colab.research.google.com).
2. Menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU** (gratis).
3. Siapkan token API Kaggle: buka kaggle.com → Account → **Create New API Token** (unduh `kaggle.json`).
4. Di Colab, klik ikon **kunci** di sidebar kiri → **Add new secret**:
   - `KAGGLE_USERNAME` = username Kaggle
   - `KAGGLE_KEY` = nilai `key` dari `kaggle.json`

Jalankan sel satu per satu dari atas ke bawah.


In [1]:
!nvidia-smi
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))


/bin/bash: line 1: nvidia-smi: command not found
TensorFlow: 2.21.0
GPU tersedia: []


## 1. Unduh dataset PlantVillage

Dataset: TensorFlow Datasets `plant_village` (54.303 citra). Kita hanya memakai empat kelas tomat: healthy, early blight, late blight, dan leaf mold.

Sumber ini tidak memerlukan akun Kaggle atau secret API.


In [2]:
!pip -q install --upgrade tensorflow-datasets importlib_resources

import shutil
from pathlib import Path
from PIL import Image
import numpy as np
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split

SELECTED = {
    "healthy": "Tomato___healthy",
    "early": "Tomato___Early_blight",
    "late": "Tomato___Late_blight",
    "mold": "Tomato___Leaf_Mold",
}

dataset, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True)
class_names_full = info.features["label"].names

selected_label_ids = {class_names_full.index(name): key for key, name in SELECTED.items()}
ROOT = Path("/content/plantdoc_data")
if ROOT.exists():
    shutil.rmtree(ROOT)

sources = {key: [] for key in SELECTED}
for image, label_id in tfds.as_numpy(dataset):
    key = selected_label_ids.get(int(label_id))
    if key is None:
        continue
    path = ROOT / "all" / key / f"{key}_{len(sources[key]):06d}.jpg"
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(image).save(path, format="JPEG", quality=95)
    sources[key].append(path)

for split in ("train", "val", "test"):
    for key in SELECTED:
        (ROOT / split / key).mkdir(parents=True, exist_ok=True)

for key, images in sources.items():
    if len(images) < 3:
        raise RuntimeError(f"Kelas {key} hanya memiliki {len(images)} citra")
    train, rest = train_test_split(images, test_size=0.30, random_state=42, shuffle=True)
    val, test = train_test_split(rest, test_size=0.50, random_state=42, shuffle=True)
    for split, group in (("train", train), ("val", val), ("test", test)):
        for src in group:
            shutil.copy2(src, ROOT / split / key / src.name)
    print(f"{key:8s} train={len(train)}  val={len(val)}  test={len(test)}")


ModuleNotFoundError: No module named 'importlib_resources'

## 2. Pilih 4 kelas & split 70% train / 15% val / 15% test

Split dilakukan **sebelum** training dengan `random_state=42`, sehingga data test benar-benar tidak pernah dilihat model. Kode di bawah mencari folder kelas secara fleksibel (mengabaikan huruf besar dan tanda baca).


In [ ]:
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

VARIANTS = {
    "healthy": [("tomato", "healthy")],
    "early":   [("tomato", "early", "blight")],
    "late":    [("tomato", "late", "blight")],
    "mold":    [("tomato", "leaf", "mold"), ("tomato", "mold")],
}
EXT = {".jpg", ".jpeg", ".png"}

def locate(variants):
    for parts in variants:
        for folder in base.rglob("*"):
            if not folder.is_dir():
                continue
            name = folder.name.lower()
            if all(part in name for part in parts):
                images = sorted(p for p in folder.rglob("*") if p.suffix.lower() in EXT)
                if len(images) > 50:
                    return folder, images
    raise FileNotFoundError(f"Folder untuk {variants} tidak ditemukan. Cek struktur dataset di {base}")

sources = {}
for key, variants in VARIANTS.items():
    folder, images = locate(variants)
    sources[key] = images
    print(f"{key:8s} {len(images):5d} citra  <- {folder}")

ROOT = Path("/content/plantdoc_data")
if ROOT.exists():
    shutil.rmtree(ROOT)
for split in ("train", "val", "test"):
    for key in VARIANTS:
        (ROOT / split / key).mkdir(parents=True, exist_ok=True)

for key, images in sources.items():
    train, rest = train_test_split(images, test_size=0.30, random_state=42, shuffle=True)
    val, test = train_test_split(rest, test_size=0.50, random_state=42, shuffle=True)
    for split, group in (("train", train), ("val", val), ("test", test)):
        for src in group:
            shutil.copy2(src, ROOT / split / key / f"{key}_{src.name}")
    print(f"{key:8s} train={len(train)}  val={len(val)}  test={len(test)}")


## 3. Bangun model MobileNetV2 (transfer learning)

Backbone ImageNet dibekukan dulu, hanya classification head yang dilatih. Augmentasi_ringan ditempatkan di dalam model agar aktif saat training dan otomatis mati saat inference.

Catatan: `image_dataset_from_directory` mengurutkan kelas secara alfabetis. Peta `class_names → web_keys` disimpan saat ekspor supaya web menerima kunci `healthy`, `early`, `late`, `mold`.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_HEAD = 8
EPOCHS_FT = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "train", image_size=IMG_SIZE, batch_size=BATCH, shuffle=True, seed=42)
val_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "val", image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
test_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "test", image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)

class_names = train_ds.class_names
print("Urutan kelas model:", class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

def infer_web_key(folder):
    name = folder.lower()
    if "healthy" in name: return "healthy"
    if "early" in name: return "early"
    if "mold" in name: return "mold"
    if "late" in name: return "late"
    return folder

web_keys = [infer_web_key(name) for name in class_names]
print("Peta ke web:", dict(zip(class_names, web_keys)))


In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.08),
], name="augment")

backbone = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet")
backbone.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = backbone(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
model.summary()


## 4. Latih

Tahap A: head hanya. Tahap B: buka 30 lapisan terakhir backbone dan fine-tuning dengan learning rate kecil. `EarlyStopping` mengambil epoch terbaik berdasarkan val_accuracy.


In [ ]:
from pathlib import Path

OUT = Path("/content/plantdoc_model")
OUT.mkdir(parents=True, exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(OUT / "best.keras"), monitor="val_accuracy",
        save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.3, patience=2, min_lr=1e-7),
]

print("=== Tahap A: classification head ===")
history_head = model.fit(train_ds, validation_data=val_ds,
                         epochs=EPOCHS_HEAD, callbacks=callbacks)

print("=== Tahap B: fine-tuning 30 lapisan terakhir ===")
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
history_ft = model.fit(train_ds, validation_data=val_ds,
                       epochs=EPOCHS_FT, callbacks=callbacks)


## 5. Evaluasi pada data test

Data test tidak dipakai saat training maupun early stopping. Metrik yang dilaporkan: Accuracy, Precision, Recall, F1-Score per kelas, dan confusion matrix.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

best = tf.keras.models.load_model(OUT / "best.keras")
loss, acc = best.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {acc*100:.2f}%")

y_true = np.concatenate([y for _, y in test_ds])
y_prob = best.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Greens", colorbar=False)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()


## 6. Ekspor model + peta label

`labels.json` berisi urutan kelas model dan kunci yang dipakai web app. Simpan ketiga file (`best.keras`, `model.tflite`, `labels.json`) ke Google Drive agar tidak hilang saat runtime di-reset.


In [ ]:
import json

converter = tf.lite.TFLiteConverter.from_keras_model(best)
(OUT / "model.tflite").write_bytes(converter.convert())

labels = {"classes": class_names, "web_keys": web_keys}
(OUT / "labels.json").write_text(json.dumps(labels, indent=2))

print("File di", OUT)
for f in sorted(OUT.glob("*")):
    print(f" - {f.name} ({f.stat().st_size/1024/1024:.2f} MB)")
print(json.dumps(labels, indent=2))


In [ ]:
# Opsional: salin ke Google Drive
from google.colab import drive
drive.mount("/content/drive")

DEST = Path("/content/drive/MyDrive/plantdoc")
DEST.mkdir(parents=True, exist_ok=True)
for f in OUT.glob("*"):
    shutil.copy2(f, DEST / f.name)
print("Tersimpan di", DEST)


## 7. Hubungkan ke web

Web (`plantdoc-web/app.js`) sudah menyediakan `MODEL_ENDPOINT`. Titik integrasinya:

1. Layani model melalui API (misal FastAPI) yang menerima `POST` multipart dengan field `file`.
2. Kembalikan JSON:
   ```json
   { "predictions": { "healthy": 0.91, "early": 0.04, "late": 0.03, "mold": 0.02 } }
   ```
3. Isi `const MODEL_ENDPOINT = "..."` di `app.js` dengan URL endpoint tersebut.

Petakan hasil `argmax` memakai `web_keys` dari `labels.json`: `web_keys[int(y_pred[0])]`. Label demo otomatis hilang ketika endpoint terisi.

Contoh potongan endpoint:

```python
import json, numpy as np, tensorflow as tf
from fastapi import FastAPI, UploadFile
from PIL import Image

model = tf.keras.models.load_model("best.keras")
labels = json.load(open("labels.json"))
app = FastAPI()

@app.post("/predict")
async def predict(file: UploadFile):
    img = Image.open(file.file).convert("RGB").resize((224, 224))
    arr = np.expand_dims(np.array(img), 0)
    probs = model.predict(arr, verbose=0)[0]
    return {"predictions": {k: float(probs[labels["web_keys"].index(k)])
                            for k in ["healthy", "early", "late", "mold"]}}
```
